# Session 5: Rules and Structured Explanations

This notebook turns fuzzy metric labels into short structured explanations. 

It sits between the fuzzy labeling step and the final narrative layer by translating labels such as “low error” or “slight overprediction” into human-readable rule outputs.

The goal is not to build a complex natural-language system. Instead, the notebook uses a few simple rules to generate consistent explanation sentences that can later be turned into a short narrative summary.


In [1]:
import pandas as pd

metrics = pd.read_csv("../src/data/session04_fuzzy_labels.csv")

### Rule Function

In [2]:
def structured_explanation(mae_label, mpe_label, mape_label, da_label):
    # Error magnitude sentence
    mae_text = {
        "low error":    "The model achieves low absolute error",
        "medium error": "The model shows moderate absolute error",
        "high error":   "The model has high absolute error",
    }.get(mae_label, f"The model shows {mae_label}")

    # Bias sentence
    mpe_text = {
        "neutral":                "with no consistent directional bias",
        "slight underprediction": "with a slight tendency to underpredict demand",
        "slight overprediction":  "with a slight tendency to overpredict demand",
    }.get(mpe_label, f"({mpe_label})")

    # Scale-free accuracy sentence
    mape_text = {
        "low mape":    "Its percentage error is low, making it reliable across different demand levels.",
        "medium mape": "Its percentage error is moderate, acceptable for operational planning.",
        "high mape":   "Its percentage error is high, limiting usefulness for demand-sensitive decisions.",
    }.get(mape_label, "")

    # Directional accuracy sentence
    da_text = {
        "low DA":    "It struggles to correctly identify whether demand will rise or fall.",
        "medium DA": "It correctly identifies the direction of demand change roughly two-thirds of the time.",
        "high DA":   "It reliably identifies the direction of demand change.",
    }.get(da_label, "")

    return f"{mae_text} {mpe_text}. {mape_text} {da_text}".strip()

In [3]:
metrics["Structured_Explanation"] = metrics.apply(
    lambda r: structured_explanation(r["MAE_Label"], r["MPE_Label"], r["MAPE_Label"], r["DA_Label"]),
    axis=1
)
metrics[["Model", "MAE_Label", "MPE_Label", "MAPE_Label", "DA_Label", "Structured_Explanation"]]

,Model,MAE_Label,MPE_Label,MAPE_Label,DA_Label,Structured_Explanation
0,Naive,high error,slight overprediction,high mape,low DA,The model has high absolute error with a sligh...
1,Seasonal Naive,medium error,neutral,medium mape,medium DA,The model shows moderate absolute error with n...
2,Linear Regression,low error,slight overprediction,medium mape,low DA,The model achieves low absolute error with a s...
3,ETS,medium error,slight overprediction,medium mape,medium DA,The model shows moderate absolute error with a...
4,HWES (damped),medium error,slight overprediction,medium mape,medium DA,The model shows moderate absolute error with a...
5,SARIMA,medium error,slight overprediction,medium mape,medium DA,The model shows moderate absolute error with a...
6,Prophet,low error,slight overprediction,low mape,medium DA,The model achieves low absolute error with a s...


In [4]:
for _, row in metrics.iterrows():
    print(f"{row['Model']}: {row['Structured_Explanation']}")


Naive: The model has high absolute error with a slight tendency to overpredict demand. Its percentage error is high, limiting usefulness for demand-sensitive decisions. It struggles to correctly identify whether demand will rise or fall.
Seasonal Naive: The model shows moderate absolute error with no consistent directional bias. Its percentage error is moderate, acceptable for operational planning. It correctly identifies the direction of demand change roughly two-thirds of the time.
Linear Regression: The model achieves low absolute error with a slight tendency to overpredict demand. Its percentage error is moderate, acceptable for operational planning. It struggles to correctly identify whether demand will rise or fall.
ETS: The model shows moderate absolute error with a slight tendency to overpredict demand. Its percentage error is moderate, acceptable for operational planning. It correctly identifies the direction of demand change roughly two-thirds of the time.
HWES (damped): The 

In [5]:
metrics.to_csv("../src/data/session05_structured_explanations.csv", index=False)

These rules are intentionally short and interpretable. They convert fuzzy labels into structured statements that describe both the size of the forecasting error and the direction of any bias.